In [1]:
import os
import cv2
import numpy as np
import random
#import math
import time # Para medir el tiempo de ejecución
#import shutil # Para borrar directorios si es necesario (opcional)

In [3]:
# --- Constantes (Asegúrate que son las finales deseadas) ---
PATCH_SIZE = 512
OVERLAP = 0.5
STRIDE = int(PATCH_SIZE * (1 - OVERLAP))
VALIDATION_THRESHOLD = 0.95
SEED = 12 # Semilla para reproducibilidad
N_FOLDS = 5
AUGMENTATION_THRESHOLD_PERCENT = 0.40 # 40% para la suma de clases 3, 4, 5

# Clases minoritarias a considerar para el aumento (AJUSTADO)
MINORITY_CLASSES_FOR_AUG = {3, 4, 5}

# --- Funciones de Utilidad (sin cambios, excepto añadir set_seed al inicio) ---
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    print(f"Semilla establecida en: {seed}")

def load_image_and_mask(image_path, mask_path):
    try:
        image = cv2.imread(image_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    except Exception as e:
        return None, None
    if image is None: return None, None
    if mask is None: return None, None

    # --- INICIO: Aplicar remapeo inicial de clases ---
    mask_remapped = mask.copy()
    mask_remapped[mask_remapped == 10] = 3
    mask_remapped[mask_remapped == 11] = 3
    mask_remapped[mask_remapped == 19] = 1
    mask_remapped[mask_remapped == 20] = 1
    mask_remapped[mask_remapped > 4] = 5 
    # --- FIN: Remapeo Inicial ---
    return image, mask_remapped

def get_rotated_bbox(mask):
    if mask is None or mask.size == 0: return None
    _, thresh = cv2.threshold(mask, 0, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return None
    try:
        all_points = np.concatenate(contours)
        rect = cv2.minAreaRect(all_points)
    except ValueError:
        try:
           largest_contour = max(contours, key=cv2.contourArea)
           rect = cv2.minAreaRect(largest_contour)
        except:
            return None
    return rect

def rotate_and_crop_roi(image, mask, rect):
    if image is None or mask is None or rect is None: return None, None
    center, (width, height), angle = rect
    if width < 1 or height < 1: return None, None
    if angle < -45.0:
        angle += 90.0
        width, height = height, width

    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    (orig_h, orig_w) = image.shape[:2]
    abs_cos = abs(rotation_matrix[0, 0])
    abs_sin = abs(rotation_matrix[0, 1])
    new_w = int(orig_h * abs_sin + orig_w * abs_cos)
    new_h = int(orig_h * abs_cos + orig_w * abs_sin)
    rotation_matrix[0, 2] += (new_w / 2) - center[0]
    rotation_matrix[1, 2] += (new_h / 2) - center[1]

    border_mode = cv2.BORDER_REFLECT_101
    border_value_img = (0,0,0)
    border_value_mask = 0

    rotated_image = cv2.warpAffine(image, rotation_matrix, (new_w, new_h),
                                   flags=cv2.INTER_LINEAR, borderMode=border_mode, borderValue=border_value_img)
    rotated_mask = cv2.warpAffine(mask, rotation_matrix, (new_w, new_h),
                                  flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=border_value_mask)

    fwidth, fheight = float(width), float(height)
    crop_x = int(round((new_w / 2.0) - (fwidth / 2.0)))
    crop_y = int(round((new_h / 2.0) - (fheight / 2.0)))
    crop_x = max(0, crop_x)
    crop_y = max(0, crop_y)
    crop_width = int(round(fwidth))
    crop_height = int(round(fheight))

    if crop_x + crop_width > new_w: crop_width = new_w - crop_x
    if crop_y + crop_height > new_h: crop_height = new_h - crop_y
    if crop_width <= 0 or crop_height <= 0: return None, None

    cropped_image = rotated_image[crop_y : crop_y + crop_height, crop_x : crop_x + crop_width]
    cropped_mask = rotated_mask[crop_y : crop_y + crop_height, crop_x : crop_x + crop_width]

    target_w, target_h = int(round(width)), int(round(height))
    if cropped_mask.shape[0] != target_h or cropped_mask.shape[1] != target_w:
         if target_w > 0 and target_h > 0:
             cropped_image = cv2.resize(cropped_image, (target_w, target_h), interpolation=cv2.INTER_LINEAR)
             cropped_mask = cv2.resize(cropped_mask, (target_w, target_h), interpolation=cv2.INTER_NEAREST)
         else:
             return None, None
    return cropped_image, cropped_mask

def calculate_patch_stats(image, mask, patch_size=PATCH_SIZE, stride=STRIDE, validation_threshold=VALIDATION_THRESHOLD, augmentation_threshold=AUGMENTATION_THRESHOLD_PERCENT):
    if image is None or image.size == 0 or mask is None or mask.size == 0: return 0, 0
    h, w = mask.shape[:2]
    if h < patch_size or w < patch_size: return 0, 0

    valid_patch_count = 0
    augmentable_patch_count = 0
    total_patch_pixels = patch_size * patch_size

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patch_mask_original = mask[y:y+patch_size, x:x+patch_size]
            non_background_pixels = np.count_nonzero(patch_mask_original)
            roi_pixel_ratio = non_background_pixels / total_patch_pixels

            if roi_pixel_ratio >= validation_threshold:
                valid_patch_count += 1
                minority_pixel_count = 0
                unique, counts = np.unique(patch_mask_original, return_counts=True)
                class_counts = dict(zip(unique, counts))
                for cls in MINORITY_CLASSES_FOR_AUG:
                    minority_pixel_count += class_counts.get(cls, 0)
                minority_ratio = minority_pixel_count / total_patch_pixels
                if minority_ratio > augmentation_threshold:
                    augmentable_patch_count += 1
    return valid_patch_count, augmentable_patch_count

# --- NUEVA FUNCIÓN: Extraer y Guardar Parches para una WSI ---
def extract_save_augment_patches_for_wsi(
    wsi_filename,
    image_input_dir,
    mask_input_dir,
    output_img_dir, # Directorio de salida para imágenes del fold
    output_mask_dir, # Directorio de salida para máscaras del fold
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    validation_threshold=VALIDATION_THRESHOLD,
    augmentation_threshold=AUGMENTATION_THRESHOLD_PERCENT
):
    """
    Procesa una WSI: carga, ROI, recorta, extrae parches, valida,
    aplica aumento espejo offline estratificado, aplica remapeo final (5->0)
    y guarda los parches originales y aumentados en las carpetas de salida.
    Devuelve el número de parches originales y aumentados guardados.
    """
    base_wsi_name = os.path.splitext(wsi_filename)[0]
    image_path = os.path.join(image_input_dir, wsi_filename)
    mask_path = os.path.join(mask_input_dir, wsi_filename)

    patches_saved_original = 0
    patches_saved_augmented = 0

    if not os.path.exists(mask_path):
        print(f"    Advertencia: No se encontró máscara para {wsi_filename}. Saltando WSI.")
        return 0, 0

    image, mask_initial_remap = load_image_and_mask(image_path, mask_path)
    if image is None or mask_initial_remap is None:
        print(f"    Advertencia: Error al cargar {wsi_filename}. Saltando WSI.")
        return 0, 0

    rect = get_rotated_bbox(mask_initial_remap)
    if not rect:
        # print(f"    Advertencia: No se encontró ROI en {wsi_filename}. Saltando WSI.")
        return 0, 0

    cropped_image, cropped_mask = rotate_and_crop_roi(image, mask_initial_remap, rect)
    if cropped_image is None or cropped_mask is None:
        # print(f"    Advertencia: Error al recortar ROI para {wsi_filename}. Saltando WSI.")
        return 0, 0

    h, w = cropped_mask.shape[:2]
    if h < patch_size or w < patch_size:
        # print(f"    Advertencia: ROI recortado para {wsi_filename} es más pequeño que el tamaño del parche.")
        return 0, 0

    total_patch_pixels = patch_size * patch_size
    patch_idx = 0 # Índice de parche por WSI

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            # --- Extraer Parche ---
            patch_image = cropped_image[y:y+patch_size, x:x+patch_size]
            patch_mask_initial = cropped_mask[y:y+patch_size, x:x+patch_size] # Todavía tiene 0,1,2,3,4,5...

            # --- Validación ---
            non_background_pixels = np.count_nonzero(patch_mask_initial)
            roi_pixel_ratio = non_background_pixels / total_patch_pixels

            if roi_pixel_ratio >= validation_threshold:
                patch_idx += 1 # Incrementar índice solo para parches válidos

                # --- Comprobar si Aumentar ---
                should_augment = False
                minority_pixel_count = 0
                unique, counts = np.unique(patch_mask_initial, return_counts=True)
                class_counts = dict(zip(unique, counts))
                for cls in MINORITY_CLASSES_FOR_AUG:
                    minority_pixel_count += class_counts.get(cls, 0)
                minority_ratio = minority_pixel_count / total_patch_pixels
                if minority_ratio > augmentation_threshold:
                    should_augment = True

                # --- Remapeo Final (5 -> 0) para el Original ---
                patch_mask_final_original = patch_mask_initial.copy()
                patch_mask_final_original[patch_mask_final_original == 5] = 0

                # --- Guardar Original ---
                orig_img_filename = f"{base_wsi_name}_patch_{patch_idx:04d}.png"
                orig_mask_filename = f"{base_wsi_name}_patch_{patch_idx:04d}.png"
                orig_img_path = os.path.join(output_img_dir, orig_img_filename)
                orig_mask_path = os.path.join(output_mask_dir, orig_mask_filename)
                try:
                    cv2.imwrite(orig_img_path, patch_image)
                    cv2.imwrite(orig_mask_path, patch_mask_final_original)
                    patches_saved_original += 1
                except Exception as e:
                    print(f"    Error guardando parche original {patch_idx} de {wsi_filename}: {e}")
                    continue # Saltar al siguiente parche si falla el guardado

                # --- Aumentar y Guardar (si aplica) ---
                if should_augment:
                    # Aplicar espejo horizontal
                    augmented_image = cv2.flip(patch_image, 1)
                    augmented_mask_initial = cv2.flip(patch_mask_initial, 1)

                    # Remapeo Final (5 -> 0) para el Aumentado
                    augmented_mask_final = augmented_mask_initial.copy()
                    augmented_mask_final[augmented_mask_final == 5] = 0

                    # Guardar aumentado
                    aug_img_filename = f"{base_wsi_name}_patch_{patch_idx:04d}_aug_hflip.png"
                    aug_mask_filename = f"{base_wsi_name}_patch_{patch_idx:04d}_aug_hflip.png"
                    aug_img_path = os.path.join(output_img_dir, aug_img_filename)
                    aug_mask_path = os.path.join(output_mask_dir, aug_mask_filename)
                    try:
                        cv2.imwrite(aug_img_path, augmented_image)
                        cv2.imwrite(aug_mask_path, augmented_mask_final)
                        patches_saved_augmented += 1
                    except Exception as e:
                        print(f"    Error guardando parche aumentado {patch_idx} de {wsi_filename}: {e}")

    return patches_saved_original, patches_saved_augmented

# --- Flujo Principal para Extracción Real ---
if __name__ == "__main__":
    print(f"Inicio del script de EXTRACCIÓN: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    start_time_total = time.time()
    set_seed(SEED)

    # --- Configuración de Rutas ---
    # ¡¡¡AJUSTA ESTAS RUTAS!!!
    base_path = "C:/Users/sorap/Documents/Johan/Universidad/Pregrado/data/"
    images_input_dir = os.path.join(base_path, "original/Images/Train/")
    masks_input_dir = os.path.join(base_path, "original/Masks/Train")
    # Directorio PADRE donde se crearán k1, k2...
    output_parent_dir = os.path.join(base_path, "patches/512x512/")

    # --- Fase 0: Listar Archivos (igual que antes) ---
    print("\n--- Fase 0: Listando Archivos ---")
    try:
        all_files = sorted([f for f in os.listdir(images_input_dir)
                        if os.path.isfile(os.path.join(images_input_dir, f)) and
                        f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff'))])
    except FileNotFoundError: print(f"Error: Dir imágenes '{images_input_dir}' no encontrado."); exit()
    if not all_files: print(f"Error: No se encontraron imágenes."); exit()
    print(f"Encontrados {len(all_files)} archivos.")

    # --- Fase 1: Pre-cálculo Conteos (Necesario para la asignación) ---
    print("\n--- Fase 1: Pre-calculando conteo de parches originales por WSI (necesario para KFold ponderado) ---")
    wsi_patch_counts = {}
    processed_count_f1 = 0
    start_time_f1 = time.time()
    for i, filename in enumerate(all_files):
        # No imprimir progreso aquí, es más rápido
        image_path = os.path.join(images_input_dir, filename)
        mask_path = os.path.join(masks_input_dir, filename)
        if not os.path.exists(mask_path): wsi_patch_counts[filename] = 0; continue
        image, mask = load_image_and_mask(image_path, mask_path)
        if image is None or mask is None: wsi_patch_counts[filename] = 0; continue
        rect = get_rotated_bbox(mask)
        if not rect: wsi_patch_counts[filename] = 0; continue
        cropped_image, cropped_mask = rotate_and_crop_roi(image, mask, rect)
        if cropped_image is None or cropped_mask is None: wsi_patch_counts[filename] = 0; continue
        orig_count, _ = calculate_patch_stats(cropped_image, cropped_mask, validation_threshold=VALIDATION_THRESHOLD)
        wsi_patch_counts[filename] = orig_count
        if orig_count > 0: processed_count_f1 += 1
    end_time_f1 = time.time()
    print(f"Fase 1 (Pre-cálculo) completada en {end_time_f1 - start_time_f1:.2f} seg.")
    wsi_data = [(wsi, count) for wsi, count in wsi_patch_counts.items() if count > 0]
    if not wsi_data: print("Error: Ninguna WSI generó parches válidos."); exit()
    print(f"Total WSIs con parches válidos: {len(wsi_data)}")
    total_original_patches_dataset = sum(count for _, count in wsi_data)
    print(f"Número total estimado de parches originales: {total_original_patches_dataset}")
    if total_original_patches_dataset == 0: print("Error: Conteo total 0."); exit()

    # --- Fase 2: Asignación Ponderada (igual que antes) ---
    print("\n--- Fase 2: Asignando WSIs a Folds (Greedy Ponderado) ---")
    start_time_f2 = time.time()
    wsi_data.sort(key=lambda item: item[1], reverse=True)
    folds_wsis = [[] for _ in range(N_FOLDS)]
    fold_patch_sums = [0] * N_FOLDS
    for wsi, count in wsi_data:
        min_sum_fold_idx = np.argmin(fold_patch_sums)
        folds_wsis[min_sum_fold_idx].append(wsi)
        fold_patch_sums[min_sum_fold_idx] += count
    end_time_f2 = time.time()
    print(f"Fase 2 (Asignación) completada en {end_time_f2 - start_time_f2:.2f} seg.")
    # (Opcional: Imprimir distribución asignada si se desea)
    for i in range(N_FOLDS):
        print(f"  Fold {i+1}: {len(folds_wsis[i])} WSIs, Suma estimada parches = {fold_patch_sums[i]}")

    # --- Fase 3: Extracción y Guardado Real por Fold ---
    print(f"\n--- Fase 3: Extrayendo, Aumentando y Guardando Parches por Fold ---")
    print(f"Directorio raíz de salida: {output_parent_dir}")
    print(f"Aplicando aumento espejo offline si Suma Clases {MINORITY_CLASSES_FOR_AUG} > {AUGMENTATION_THRESHOLD_PERCENT*100:.1f}%")
    start_time_f3 = time.time()
    grand_total_orig_saved = 0
    grand_total_aug_saved = 0

    for fold_idx in range(N_FOLDS):
        fold_num = fold_idx + 1
        current_fold_wsis = folds_wsis[fold_idx]
        print(f"\n  --- Procesando Fold {fold_num}/{N_FOLDS} ({len(current_fold_wsis)} WSIs) ---")

        # Crear directorios de salida para este fold
        fold_dir = os.path.join(output_parent_dir, f"k{fold_num}")
        img_out_dir = os.path.join(fold_dir, "Images")
        mask_out_dir = os.path.join(fold_dir, "Masks")
        # # Opcional: Borrar directorio existente para empezar limpio
        # if os.path.exists(fold_dir):
        #     print(f"    Eliminando directorio existente: {fold_dir}")
        #     shutil.rmtree(fold_dir)
        os.makedirs(img_out_dir, exist_ok=True)
        os.makedirs(mask_out_dir, exist_ok=True)
        print(f"    Directorio de salida: {fold_dir}")

        fold_total_orig_saved_count = 0
        fold_total_aug_saved_count = 0
        processed_wsis_count = 0

        # Procesar cada WSI asignada a este fold
        for i, filename in enumerate(current_fold_wsis):
            #print(f"      Procesando WSI {i+1}/{len(current_fold_wsis)}: {filename}...")
            orig_saved, aug_saved = extract_save_augment_patches_for_wsi(
                filename,
                images_input_dir,
                masks_input_dir,
                img_out_dir,
                mask_out_dir,
                patch_size=PATCH_SIZE,
                stride=STRIDE,
                validation_threshold=VALIDATION_THRESHOLD,
                augmentation_threshold=AUGMENTATION_THRESHOLD_PERCENT
            )
            if orig_saved > 0: # Contar como procesada si guardó al menos un parche original
                processed_wsis_count += 1
            fold_total_orig_saved_count += orig_saved
            fold_total_aug_saved_count += aug_saved

        print(f"    Fold {fold_num} completado:")
        print(f"      WSIs procesadas: {processed_wsis_count}/{len(current_fold_wsis)}")
        print(f"      Parches Originales Guardados: {fold_total_orig_saved_count}")
        print(f"      Parches Aumentados Guardados: {fold_total_aug_saved_count}")
        print(f"      Total Parches Guardados en Fold: {fold_total_orig_saved_count + fold_total_aug_saved_count}")
        grand_total_orig_saved += fold_total_orig_saved_count
        grand_total_aug_saved += fold_total_aug_saved_count

    end_time_f3 = time.time()
    print(f"\nFase 3 (Extracción y Guardado) completada en {end_time_f3 - start_time_f3:.2f} segundos.")

    # --- Fase 4: Resumen Final del Guardado ---
    print("\n" + "="*40)
    print("--- RESUMEN FINAL - PARCHES GUARDADOS ---")
    print("="*40)
    print(f"Total Parches Originales Guardados (Todos los Folds): {grand_total_orig_saved}")
    print(f"Total Parches Aumentados Guardados (Todos los Folds): {grand_total_aug_saved}")
    print(f"Total General Parches Guardados (Originales + Aumentados): {grand_total_orig_saved + grand_total_aug_saved}")
    print("-" * 40)
    # Verificar si el total original guardado coincide con el pre-cálculo
    if grand_total_orig_saved != total_original_patches_dataset:
         print(f"  ADVERTENCIA: Total originales guardados ({grand_total_orig_saved}) difiere del pre-cálculo ({total_original_patches_dataset}).")
         print(f"               Esto puede ocurrir si algunas WSIs fallaron durante la Fase 3 pero no en la Fase 1.")
    print("Verifica las carpetas k1, k2, k3, k4, k5 en:")
    print(f"  {output_parent_dir}")
    print("="*40)


    end_time_total = time.time()
    print(f"\nScript completo en {end_time_total - start_time_total:.2f} segundos.")

Inicio del script de EXTRACCIÓN: 2025-04-25 01:43:55
Semilla establecida en: 12

--- Fase 0: Listando Archivos ---
Encontrados 116 archivos.

--- Fase 1: Pre-calculando conteo de parches originales por WSI (necesario para KFold ponderado) ---
Fase 1 (Pre-cálculo) completada en 226.24 seg.
Total WSIs con parches válidos: 116
Número total estimado de parches originales: 19396

--- Fase 2: Asignando WSIs a Folds (Greedy Ponderado) ---
Fase 2 (Asignación) completada en 0.02 seg.
  Fold 1: 23 WSIs, Suma estimada parches = 3880
  Fold 2: 23 WSIs, Suma estimada parches = 3879
  Fold 3: 23 WSIs, Suma estimada parches = 3878
  Fold 4: 24 WSIs, Suma estimada parches = 3880
  Fold 5: 23 WSIs, Suma estimada parches = 3879

--- Fase 3: Extrayendo, Aumentando y Guardando Parches por Fold ---
Directorio raíz de salida: C:/Users/sorap/Documents/Johan/Universidad/Pregrado/data/patches/512x512/
Aplicando aumento espejo offline si Suma Clases {3, 4, 5} > 40.0%

  --- Procesando Fold 1/5 (23 WSIs) ---
   

KeyboardInterrupt: 